# Simulation 2 Control Cued Behavioral Figure

Figure-only renderer for the Simulation 2 control figure. This notebook reads aggregate phase totals and maximal-contrast summaries; it does not rerun simulations or save full simulated recall sequences.

In [1]:
import csv
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from jaxcmr.helpers import find_project_root

from selective_interference_v2 import PHASE_COLORS, save_figure

warnings.filterwarnings("ignore")
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]


In [2]:
# --- figure configuration ---
PROJECT_ROOT = ""
FIGURE_DIR = "figures"
FIGURE_STR = "simulation2_control_cued"

PHASE_TOTALS_PATH = ""
COST_PATH = ""

TASK_MCF_VALUES = [1.0, 2.0]
TASK_MCF_LABELS = {
    1.0: "Weak task encoding",
    2.0: "Strong task encoding",
}
REMINDER_CONDITIONS = ["No reminder", "With reminder"]
BAR_REMINDER_LABELS = {
    "No reminder": "No reminder",
    "With reminder": "Film reminder",
}
COST_REGIMES = {
    "unguided": "Unguided recall",
    "start_only": "Deliberate recall: start context",
    "start_monitoring": "Deliberate recall: start context + monitoring",
}


In [3]:
# Parameters
FIGURE_DIR = "figures"
FIGURE_STR = "simulation2_control_cued"


In [4]:
project_root = Path(PROJECT_ROOT).expanduser() if PROJECT_ROOT else find_project_root()
figure_dir = Path(FIGURE_DIR).expanduser() if FIGURE_DIR else None
if figure_dir is not None and not figure_dir.is_absolute():
    figure_dir = project_root / figure_dir

if figure_dir is None or not FIGURE_STR:
    raise ValueError("FIGURE_DIR and FIGURE_STR are required for figure-only rendering.")

phase_totals_path = Path(PHASE_TOTALS_PATH).expanduser() if PHASE_TOTALS_PATH else figure_dir / f"{FIGURE_STR}_phase_totals.csv"
cost_path = Path(COST_PATH).expanduser() if COST_PATH else figure_dir / f"{FIGURE_STR}_maximal_interference_cost.csv"

print(f"Reading phase totals from {phase_totals_path}")
print(f"Reading maximal-contrast costs from {cost_path}")


Reading phase totals from /Users/jordangunn/workspace/selective_interference_v2/figures/simulation2_control_cued_phase_totals.csv
Reading maximal-contrast costs from /Users/jordangunn/workspace/selective_interference_v2/figures/simulation2_control_cued_maximal_interference_cost.csv


In [5]:
PANEL_LETTER_FONTSIZE = 16
PANEL_TITLE_FONTSIZE = 13
STRUCTURAL_FONTSIZE = 11
SUPPORT_FONTSIZE = 10

FILM_BAR_COLOR = PHASE_COLORS["film"]
FILM_BAR_PALE = "#E8F1FF"
FILM_BAR_EDGE = PHASE_COLORS["film"]

CONTRAST_LABELS = {
    "benign": "No reminder + weak task encoding",
    "hardest": "Film reminder + strong task encoding",
}


def read_rows(path):
    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))


def coerce_phase_row(row):
    row = dict(row)
    row["start_drift_scale"] = float(row["start_drift_scale"])
    row["rejected_recall_drift_scale"] = float(row["rejected_recall_drift_scale"])
    row["task_mcf_scale"] = float(row["task_mcf_scale"])
    row["recall_probability_mass"] = float(row["recall_probability_mass"])
    return row


def coerce_cost_row(row):
    row = dict(row)
    row["start_drift_scale"] = float(row["start_drift_scale"])
    row["rejected_recall_drift_scale"] = float(row["rejected_recall_drift_scale"])
    row["benign_film_recall"] = float(row["benign_film_recall"])
    row["hardest_film_recall"] = float(row["hardest_film_recall"])
    row["maximal_interference_cost"] = float(row["maximal_interference_cost"])
    row["maximal_interference_percent"] = float(row["maximal_interference_percent"])
    return row


phase_rows = [coerce_phase_row(row) for row in read_rows(phase_totals_path)]
cost_rows = [coerce_cost_row(row) for row in read_rows(cost_path)]
cost_rows_by_regime = {row["retrieval_regime"]: row for row in cost_rows}

missing = [regime for regime in COST_REGIMES.values() if regime not in cost_rows_by_regime]
if missing:
    raise ValueError(f"Missing retrieval regimes in cost file: {missing}")

unguided_row = cost_rows_by_regime[COST_REGIMES["unguided"]]
start_only_row = cost_rows_by_regime[COST_REGIMES["start_only"]]
start_monitoring_row = cost_rows_by_regime[COST_REGIMES["start_monitoring"]]
selected_monitoring_kappa = start_monitoring_row["rejected_recall_drift_scale"]
selected_start_drift = start_only_row["start_drift_scale"]

RETRIEVAL_SETTINGS = [
    {
        "label": "Unguided\nrecall",
        "start_drift_scale": unguided_row["start_drift_scale"],
        "rejected_recall_drift_scale": unguided_row["rejected_recall_drift_scale"],
    },
    {
        "label": "Target\nmonitoring",
        "start_drift_scale": unguided_row["start_drift_scale"],
        "rejected_recall_drift_scale": selected_monitoring_kappa,
    },
    {
        "label": "Start-of-film\nreinstatement",
        "start_drift_scale": selected_start_drift,
        "rejected_recall_drift_scale": start_only_row["rejected_recall_drift_scale"],
    },
    {
        "label": "Reinstatement\n+ monitoring",
        "start_drift_scale": selected_start_drift,
        "rejected_recall_drift_scale": selected_monitoring_kappa,
    },
]


def add_panel_label(axis, label, x=-0.16, y=1.13):
    axis.text(
        x,
        y,
        label,
        transform=axis.transAxes,
        fontsize=PANEL_LETTER_FONTSIZE,
        fontweight="bold",
        va="top",
        ha="left",
    )


def phase_total(start_drift_scale, kappa, reminder_condition, task_scale, phase):
    for row in phase_rows:
        if (
            np.isclose(row["start_drift_scale"], start_drift_scale)
            and np.isclose(row["rejected_recall_drift_scale"], kappa)
            and row["reminder_condition"] == reminder_condition
            and np.isclose(row["task_mcf_scale"], task_scale)
            and row["phase"] == phase
        ):
            return row["recall_probability_mass"]
    return np.nan


def endpoint_values(setting):
    benign = phase_total(
        setting["start_drift_scale"],
        setting["rejected_recall_drift_scale"],
        "No reminder",
        float(TASK_MCF_VALUES[0]),
        "film",
    )
    hardest = phase_total(
        setting["start_drift_scale"],
        setting["rejected_recall_drift_scale"],
        "With reminder",
        float(TASK_MCF_VALUES[-1]),
        "film",
    )
    reduction = benign - hardest
    percent = 100 * reduction / benign if benign > 0 else np.nan
    return benign, hardest, reduction, percent


def style_axis(axis):
    axis.tick_params(labelsize=SUPPORT_FONTSIZE)
    axis.set_axisbelow(True)
    axis.grid(axis="y", color="#E7EBF0", linewidth=0.45, alpha=0.45, zorder=0)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)



In [6]:
x = np.arange(len(RETRIEVAL_SETTINGS))
width = 0.32
endpoint_rows = [endpoint_values(setting) for setting in RETRIEVAL_SETTINGS]
benign_values = [row[0] for row in endpoint_rows]
hardest_values = [row[1] for row in endpoint_rows]
preserved_values = [100 * row[1] / row[0] if row[0] > 0 else np.nan for row in endpoint_rows]
retrieval_labels = [setting["label"] for setting in RETRIEVAL_SETTINGS]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11.3, 4.2),
    gridspec_kw={"wspace": 0.30, "left": 0.080, "right": 0.985, "top": 0.900, "bottom": 0.265},
)
axis_output, axis_preserved = axes

axis_output.bar(
    x - width / 2,
    benign_values,
    width=width,
    color=FILM_BAR_PALE,
    edgecolor=FILM_BAR_EDGE,
    linewidth=1.1,
    label=CONTRAST_LABELS["benign"],
    zorder=2,
)
axis_output.bar(
    x + width / 2,
    hardest_values,
    width=width,
    color=FILM_BAR_COLOR,
    edgecolor=FILM_BAR_EDGE,
    linewidth=1.1,
    label=CONTRAST_LABELS["hardest"],
    zorder=2,
)
axis_output.set_title("Film recall by condition and retrieval setting", fontsize=PANEL_TITLE_FONTSIZE, pad=8)
axis_output.set_xticks(x)
axis_output.set_xticklabels(retrieval_labels)
axis_output.set_ylabel("Mean film items recalled", fontsize=STRUCTURAL_FONTSIZE)
axis_output.set_ylim(0, max(benign_values + hardest_values) * 1.16)
style_axis(axis_output)
add_panel_label(axis_output, "A")

axis_preserved.bar(
    x,
    preserved_values,
    width=0.46,
    color=FILM_BAR_COLOR,
    edgecolor=FILM_BAR_EDGE,
    linewidth=1.1,
    zorder=2,
)
axis_preserved.set_title("Film recall preserved under high interference", fontsize=PANEL_TITLE_FONTSIZE, pad=8)
axis_preserved.set_xticks(x)
axis_preserved.set_xticklabels(retrieval_labels)
axis_preserved.set_ylabel("Film recall preserved (%)", fontsize=STRUCTURAL_FONTSIZE)
axis_preserved.set_ylim(0, 100)
axis_preserved.set_yticks([0, 25, 50, 75, 100])
style_axis(axis_preserved)
add_panel_label(axis_preserved, "B")

handles, labels = axis_output.get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    frameon=False,
    fontsize=SUPPORT_FONTSIZE,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.035),
    ncol=2,
    columnspacing=1.6,
    handlelength=1.6,
)

save_figure(
    str(figure_dir) if figure_dir is not None else "",
    FIGURE_STR,
    "behavioral",
)


